<a href="https://colab.research.google.com/github/kashu1337/ML-for-physicists/blob/main/Machine_Learning_for_Physicists_Homework_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning for Physicists - Homework 02

In [2]:
#from numpy import array, zeros, exp, random, dot, shape, reshape, meshgrid, linspace
import numpy as np

import matplotlib.pyplot as plt # for plotting
import matplotlib
matplotlib.rcParams['figure.dpi']=300 # highres display

# for subplots within subplots:
from matplotlib import gridspec

# for nice inset colorbars: (approach changed from lecture 1 'Visualization' notebook)
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
# for updating display
# (very simple animation)
from IPython.display import clear_output
from time import sleep

### (1) Carefully study the backpropagation algorithm, on paper and in the program.

Let's explain how the theory works in good detail. Our NN is a sequence

\begin{align*}
L_0 \stackrel{\xi(1,0)}{\to} L_1 \stackrel{\xi(2,1)}{\to} ... \stackrel{\xi(N,N-1)}{\to} L_N
\end{align*}

where we take our neuron spaces at each layer to be

\begin{align*}
L_N := \mathbb{R}^{s_N}
\end{align*}

viewed as linear spaces, and each map between layers is

\begin{align*}
\xi(n+1,n)(y^j e_j) &= f(n+1,n) \bigg[ (W(n+1,n)^k_j y^j + b(n+1,n)^k)e_k \bigg]
\\
&=: f(n+1,n) (z^k e_k)
\end{align*}

where $f(n+1,n): L_{n+1} \to L_{n+1}$ is a non-linear function. A lot of the time, the $j$'th component of $f$ is only a function of $z^j$, but not always. Let us assume this is the case. An example is the sigmoid

\begin{align*}
\sigma(\vec{z})^l = \frac{1}{1+\exp(-z^l)}
\end{align*}

whereas an example that breaks this assumption is the softmax

\begin{align*}
s(\vec{z})^l = \frac{\exp(z^l)}{\sum_{m} \exp(z^{m})}
\end{align*}

where all the input coefficients contribute to a given output coefficient. The assumption will make backpropagation easier to write down.

Fixing an $f$, the NN belongs to a family of functions where the weight matrices and bias vectors come together to form a point $\theta \in \Theta$ in a large parameter space. We denote $\xi(N,N-1) \circ ... \circ \xi(1,0) (x) = F_x(\theta) \in L_N$ as the output of the full NN for a choice of parameters.

As explained in HW1, we can train the NN on a batch by performing a gradient descent that updates the parameters $\theta$. This requires us to know the derivatives of the cost function. If I was stupid, I would ask the computer to numerically compute derivatives for me. However, numerical differentiation does not take advantage of any theorems and therefore takes ages. ML would not work practically in that case. Backprop is a fancy word applied to the obvious idea that the degrees of freedom in $\theta$ are effectively coupled using the cost function and the NN's layer structure, and so we can use the chain rule to make things easier for the computer. Namely, the computer won't have to do any numerical differentiation at all, but rather do a series of linear operations via if-else loops. This will be no slower than just doing a full feed-forward pass $x \mapsto F_x(\theta) \in L_N$, and that's the beauty of it all.

Say I have a batch $B$ in my input probability distribution. We will be interested in averages of the quadratic cost over this batch: $\langle C_x(\theta) \rangle_{x \in B} = \sum_{x \in B} p_x C_x(\theta)$. The derivative of this cost is then specified by the derivative of $C_x(\theta)$ for some $x \in L_0$. We compute this as

\begin{align*}
\partial_k C_x(\theta) &= \partial_k \frac{1}{2} |F_x(\theta) - T(x)|^2
\\
&= (F_x(\theta) - T(x)) \cdot_{\mathbb{R}^{s_N}} \partial_k F_x(\theta).
\end{align*}

The Euclidean dot product is used here. We can view $F_x(\theta)$ as the output vector, so it has components $y^j_{(N)}$. We thus have

\begin{align*}
\partial_k C_x(\theta) = \sum_j (y^j_{(N)} - T(x)^j) \cdot f'(z_{(N)}^j) \cdot \partial_k z^j_{(N)}
\end{align*}

where I now explicitly indicate the sum. We can actually compute these $\partial_k z^j_{(N)}$ derivatives exactly. If the parameter $\theta^k$ is at a layer $n$, then we can repeatedly apply the chain rule at some layer $m > n$ to get

\begin{align*}
\partial_k z^l_{(m)} = \bigg[ \hat{M}(m,m-1) \circ \hat{M}(m-1,m-2) \circ ... \circ \hat{M}(n+1,n) \cdot \partial_k \vec{z}_{(n)} \bigg]^l
\end{align*}

where $M(n,n-1)^l_m := W(n,n-1)^l_m \cdot f'(z_{(n-1)}^m)$. If $\theta^k$ is the weight at layer $n$, the final derivative is just

\begin{align*}
\frac{\partial}{\partial W(n,n-1)^a_b} z_{(n)}^l = \delta^{l,a} y_{(n-1)}^b
\end{align*}

and if $\theta^k$ is the bias at that layer,

\begin{align*}
\frac{\partial}{\partial b(n,n-1)^a} z_{(n)}^l = \delta^{a,l}.
\end{align*}

So putting all this together, we have the following way to compute all the derivatives of the cost function:

1) For $\theta^k$ describing a parameter in the final layer $(N,N-1)$, we have

\begin{align*}
\partial_k C_x(\theta) &= \sum_j (y^j_{(N)} - T(x)^j) \cdot f'(z^j_{(N)}) \cdot \partial_k z^j_{(N)}
\\
&=: \sum_j \Delta^j_{(N)} \cdot \partial_k z^j_{(N)}
\\
&= \vec{\Delta}_{(N)} \cdot \partial_k \vec{z}_{(N)}
\end{align*}

and we can compute the derivatives directly at this, either giving us an output neuron at layer $N-1$, or some constants.

2) For $\theta^k$ describing a parameter in the penultimate layer $(N-1,N-2)$, we have

\begin{align*}
\partial_k C_x(\theta)&= \vec{\Delta} \cdot \partial_k \vec{z}_{(N)}
\\
&= \vec{\Delta}_{(N)} \cdot \hat{M}(N,N-1) \partial_k \vec{z}_{(N-1)}
\\
&= \vec{\Delta}_{(N-1)} \cdot \partial_k \vec{z}_{(N-1)}
\end{align*}

where we can again easily compute the vector on the right hand side, and

\begin{align*}
\vec{\Delta}_{(N-1)}^j = \sum_k \Delta^{k}_{(N)} \cdot M(N,N-1)^{kj}.
\end{align*}

3) Repeat this process, starting from the $\theta$'s that live at the top layer and going down. This will give you a list of all cost function derivatives without the computer actually having to do any numerical differentiation.


That was a conceptual description. The actual backprop algo is then

1) Initialise a starting value of $\theta \in \Theta$, and fix some $x \in L_0$. Compute the *deviation vector* $\Delta_{(N)}^j = (y^j_{(N)} - T(x)^j) \cdot f'(z^j_{(N)})$ at layer $N$.

2) Compute all the derivatives at layer $N$, which is to say take those $\theta^k$ that describe parameters at layer $N$ and compute $\vec{\Delta}_{(N)} \cdot \partial_k \vec{z}_{(N)}$ where the RHS vector is either some $y_{(N-1)}$ neuron, or a constant.

3) Now compute the deviation vector $\Delta^{(j)}_{(N-1)} = \sum_k \Delta^k_{(N)} M(N,N-1)^{kj} = \sum_{k,j} \Delta^k_{(N)} W(N,N-1)^{kj} f'(z^{j}_{(N-1)})$.

4) The derivatives at layer $N-1$ are now obtained via $\partial_k C_x(\theta) = \vec{\Delta}_{(N-1)} \cdot \partial_k \vec{z}_{(N-1)}$

5) Repeat until all derivatives are obtained.

That's all. Now that we have a very efficient way to compute derivatives, we can flow in $\Theta$ according to

\begin{align}
\delta \theta^k \approx - \eta \sum_{x \in B} p_x \partial^k C_x(\theta).
\end{align}

Then training data lets us do the descent to get a really good value of $\theta_\star$, and then we use the resulting $F_x(\theta_\star)$ as our AI output. Let's now implement this below.

In [ ]:
# @title
def net_f_df(z,activation):
    # return both value f(z) and derivative f'(z) as a pair. these are our activation functions
    if activation=='sigmoid':
        return([1/(1+np.exp(-z)),
                1/((1+np.exp(-z))*(1+np.exp(z))) ])
    elif activation=='jump': # cheating a bit here: replacing f'(z)=delta(z) by something smooth. its a step function
        return([np.array(z>0,dtype='float'),
                10.0/((1+np.exp(-10*z))*(1+np.exp(10*z))) ] )
    elif activation=='linear':
        return([z,
                1.0])
    elif activation=='reLU': # its clever, as z>0 is an integer 0 or 1 which we calculate with! well, the previous version would use (z>0)*z as f and (z>0) as f'. now its
        return([(z>0)*z,
                (z>0)*1.0
               ])

def forward_step(y,w,b,activation):
    """
    Go from one layer to the next, given a
    weight matrix w (shape [n_neurons_in,n_neurons_out])
    a bias vector b (length n_neurons_out)
    and the values of input neurons y_in
    (shape [batchsize,n_neurons_in])

    returns the values of the output neurons in the next layer
    (shape [batchsize, n_neurons_out])
    """
    # calculate values in next layer, from input y
    z=np.dot(y,w)+b # w=weights, b=bias vector for next layer. dot product backwards to do parallel batch processing in a python-efficient way
    return(net_f_df(z,activation)) # apply nonlinearity and return result

def apply_net(y_in): # one forward pass through the network
    global Weights, Biases, NumLayers, Activations
    global y_layer, df_layer # for storing y-values and df/dz values, since we want these to do backprop in a bit

    y=np.copy(y_in) # start with input values
    y_layer[0]=np.copy(y)
    for j in range(NumLayers): # loop through all layers
        # j=0 corresponds to the first layer above the input i.e. L_1
        y,df=forward_step(y,Weights[j],Biases[j],Activations[j]) # one step
        df_layer[j]=np.copy(df) # store f'(z) values [needed later in backprop]
        y_layer[j+1]=np.copy(y) # store f(z) values [also needed in backprop]
    return(y) #this output is the vector in L_N from a forward pass with input y_in.

def apply_net_simple(y_in): # one forward pass through the network
    # no storage for backprop (this is used for simple tests)
    global Weights, Biases, NumLayers, Activations

    y=y_in # start with input values
    for j in range(NumLayers): # loop through all layers
        # j=0 corresponds to the first layer above the input
        y,df=forward_step(y,Weights[j],Biases[j],Activations[j]) # one step
    return(y)

def backward_step(delta,w,df):
    # delta at layer N, of batchsize x layersize(N))
    # w between N-1 and N [layersize(N-1) x layersize(N) matrix]
    # df = df/dz at layer N-1, of batchsize x layersize(N-1)
    return( np.dot(delta,np.transpose(w))*df ) #this is essentially finding delta_{N-1} from multiplying delta_N with W matrices! see theory above

def backprop(y_target): # one full backward pass through the network
    # the result will be the 'dw_layer' matrices that contain
    # the derivatives of the cost function with respect to
    # the corresponding weight - we then use these numbers to gradient descend
    global y_layer, df_layer, Weights, Biases, NumLayers
    global dw_layer, db_layer # dCost/dw and dCost/db (w,b=weights,biases)

    batchsize=np.shape(y_target)[0]
    delta=(y_layer[-1]-y_target)*df_layer[-1]
    dw_layer[-1]=np.dot(np.transpose(y_layer[-2]),delta)/batchsize
    db_layer[-1]=delta.sum(0)/batchsize
    for j in range(NumLayers-1):
        delta=backward_step(delta,Weights[-1-j],df_layer[-2-j])
        dw_layer[-2-j]=np.dot(np.transpose(y_layer[-3-j]),delta)/batchsize # batchsize was missing in old code?
        db_layer[-2-j]=delta.sum(0)/batchsize

def gradient_step(eta): # update weights & biases via gradient descent now that we know the derivatives of C_x(theta). Will do stochastic descent since we have a batch of x's to begin with that we computed the above for! ie we give the NN a batch of inputs x and the corresponding outputs (target function). I think the inputs are considered uniformly distributed so far, which is pretty understandable.
    global dw_layer, db_layer, Weights, Biases

    for j in range(NumLayers):
        Weights[j]-=eta*dw_layer[j]
        Biases[j]-=eta*db_layer[j]

def train_net(y_in,y_target,eta): # one full training batch
    # y_in is an array of size batchsize x (input-layer-size)
    # y_target is an array of size batchsize x (output-layer-size)
    # eta is the stepsize for the gradient descent
    global y_out_result

    y_out_result=apply_net(y_in)
    backprop(y_target)
    gradient_step(eta)
    cost=0.5*((y_target-y_out_result)**2).sum()/np.shape(y_in)[0]
    return(cost) #the function returns the costs due to the outputs of the batch being off the target

def init_layer_variables(weights,biases,activations):
    global Weights, Biases, NumLayers, Activations
    global LayerSizes, y_layer, df_layer, dw_layer, db_layer

    Weights=weights
    Biases=biases
    Activations=activations
    NumLayers=len(Weights)

    LayerSizes=[2]
    for j in range(NumLayers):
        LayerSizes.append(len(Biases[j]))

    y_layer=[[] for j in range(NumLayers+1)]
    df_layer=[[] for j in range(NumLayers)]
    dw_layer=[np.zeros([LayerSizes[j],LayerSizes[j+1]]) for j in range(NumLayers)]
    db_layer=[np.zeros(LayerSizes[j+1]) for j in range(NumLayers)]
#this just creates a space for us to put our updated weights into i think

### (2) Visualize the training of a multi-layer network for some interesting function! Explore reLU vs sigmoid!

We can now use these functions to train a neural network. We want to visualise the process, so it is best to set $s_0=2$ and $s_N=1$ so that $F_{-}(\theta):\mathbb{R}^2 \to \mathbb{R}$ is viewed as a colour map. The relevant plotting stuff is developed below.

In [ ]:
# @title
# visualization routines:

# some internal routines for plotting the network:
def plot_connection_line(ax,X,Y,W,vmax=1.0,linewidth=3):
    t=np.linspace(0,1,20)
    if W>0:
        col=[0,0.4,0.8]
    else:
        col=[1,0.3,0]
    ax.plot(X[0]+(3*t**2-2*t**3)*(X[1]-X[0]),Y[0]+t*(Y[1]-Y[0]),
           alpha=abs(W)/vmax,color=col,
           linewidth=linewidth)

def plot_neuron_alpha(ax,X,Y,B,size=100.0,vmax=1.0):
    if B>0:
        col=[0,0.4,0.8]
    else:
        col=[1,0.3,0]
    ax.scatter([X],[Y],marker='o',c=col,alpha=abs(B)/vmax,s=size,zorder=10)

def plot_neuron(ax,X,Y,B,size=100.0,vmax=1.0):
    if B>0:
        col=[0,0.4,0.8]
    else:
        col=[1,0.3,0]
    ax.scatter([X],[Y],marker='o',c=col,s=size,zorder=10)

def visualize_network(weights,biases,activations,
                      M=100,y0range=[-1,1],y1range=[-1,1],
                     size=400.0, linewidth=5.0,
                     weights_are_swapped=False,
                    layers_already_initialized=False,
                      plot_cost_function=None,
                      current_cost=None, cost_max=None, plot_target=None
                     ):
    """
    Visualize a neural network with 2 input
    neurons and 1 output neuron (plot output vs input in a 2D plot)

    weights is a list of the weight matrices for the
    layers, where weights[j] is the matrix for the connections
    from layer j to layer j+1 (where j==0 is the input)

    weights[j][m,k] is the weight for input neuron k going to output neuron m
    (note: internally, m and k are swapped, see the explanation of
    batch processing in lecture 2)

    biases[j] is the vector of bias values for obtaining the neurons in layer j+1
    biases[j][k] is the bias for neuron k in layer j+1

    activations is a list of the activation functions for
    the different layers: choose 'linear','sigmoid',
    'jump' (i.e. step-function), and 'reLU'

    M is the resolution (MxM grid)

    y0range is the range of y0 neuron values (horizontal axis)
    y1range is the range of y1 neuron values (vertical axis)
    """
    if not weights_are_swapped:
        swapped_weights=[]
        for j in range(len(weights)):
            swapped_weights.append(np.transpose(weights[j]))
    else:
        swapped_weights=weights

    y0,y1=np.meshgrid(np.linspace(y0range[0],y0range[1],M),np.linspace(y1range[0],y1range[1],M))
    y_in=np.zeros([M*M,2])
    y_in[:,0]=y0.flatten()
    y_in[:,1]=y1.flatten()

    # if we call visualization directly, we still
    # need to initialize the 'Weights' and other
    # global variables; otherwise (during training)
    # all of this has already been taken care of:
    if not layers_already_initialized:
        init_layer_variables(swapped_weights,biases,activations)
    y_out=apply_net_simple(y_in)

    if plot_cost_function is None:
        fig,ax=plt.subplots(ncols=2,nrows=1,figsize=(8,4))
    else:
        fig=plt.figure(figsize=(8,4))
        gs_top = gridspec.GridSpec(nrows=1, ncols=2)
        gs_left = gridspec.GridSpecFromSubplotSpec(nrows=2, ncols=1, subplot_spec=gs_top[0], height_ratios=[1.0,0.3])
        ax=[ fig.add_subplot(gs_left[0]),
            fig.add_subplot(gs_top[1]),
           fig.add_subplot(gs_left[1]) ]
        # ax[0] is network
        # ax[1] is image produced by network
        # ax[2] is cost function subplot

    # plot the network itself:

    # positions of neurons on plot:
    posX=[[-0.5,+0.5]]; posY=[[0,0]]
    vmax=0.0 # for finding the maximum weight
    vmaxB=0.0 # for maximum bias
    for j in range(len(biases)):
        n_neurons=len(biases[j])
        posX.append(np.array(range(n_neurons))-0.5*(n_neurons-1))
        posY.append(np.full(n_neurons,j+1))
        vmax=np.maximum(vmax,np.max(np.abs(weights[j])))
        vmaxB=np.maximum(vmaxB,np.max(np.abs(biases[j])))

    # plot connections
    for j in range(len(biases)):
        for k in range(len(posX[j])):
            for m in range(len(posX[j+1])):
                plot_connection_line(ax[0],[posX[j][k],posX[j+1][m]],
                                     [posY[j][k],posY[j+1][m]],
                                     swapped_weights[j][k,m],vmax=vmax,
                                    linewidth=linewidth)

    # plot neurons
    for k in range(len(posX[0])): # input neurons (have no bias!)
        plot_neuron(ax[0],posX[0][k],posY[0][k],
                   vmaxB,vmax=vmaxB,size=size)
    for j in range(len(biases)): # all other neurons
        for k in range(len(posX[j+1])):
            plot_neuron(ax[0],posX[j+1][k],posY[j+1][k],
                       biases[j][k],vmax=vmaxB,size=size)

    ax[0].axis('off')

    # now: the output of the network
    img=ax[1].imshow(np.reshape(y_out,[M,M]),origin='lower',
                    extent=[y0range[0],y0range[1],y1range[0],y1range[1]])
    ax[1].set_xlabel(r'$y_0$')
    ax[1].set_ylabel(r'$y_1$')

#     axins1 = inset_axes(ax[1],
#                     width="40%",  # width = 50% of parent_bbox width
#                     height="5%",  # height : 5%
#                     loc='upper right',
#                        bbox_to_anchor=[0.3,0.4])

#    axins1 = ax[1].inset_axes([0.5,0.8,0.45,0.1])
    axins1 = ax[1].inset_axes([0.25, 0.1, 0.5, 0.05])

    #axins1 = plt.axes([0, 0, 1, 1])
    #ip = InsetPosition(ax[1], [0.25, 0.1, 0.5, 0.05])
    #axins1.set_axes_locator(ip)

    imgmin=np.min(y_out)
    imgmax=np.max(y_out)
    color_bar=fig.colorbar(img, cax=axins1, orientation="horizontal",ticks=np.linspace(imgmin,imgmax,3))
    cbxtick_obj = plt.getp(color_bar.ax.axes, 'xticklabels')
    plt.setp(cbxtick_obj, color="white")
    axins1.xaxis.set_ticks_position("bottom")

    if plot_target is not None:
        axins2 = ax[1].inset_axes([0.75, 0.75, 0.2, 0.2])
        axins2.imshow(plot_target, origin='lower')
        axins2.set_xticks([])
        axins2.set_yticks([])

    if plot_cost_function is not None:
        ax[2].plot(plot_cost_function)
        ax[2].set_ylim([0.0,cost_max])
        ax[2].set_yticks([0.0,cost_max])
        ax[2].set_yticklabels(["0",'{:1.2e}'.format(cost_max)])
        if current_cost is not None:
            ax[2].text(0.9, 0.9, 'cost={:1.2e}'.format(current_cost), horizontalalignment='right',
                       verticalalignment='top', transform=ax[2].transAxes)

    plt.show()

def visualize_network_training(weights,biases,activations,
                               target_function,
                               num_neurons=None,
                               weight_scale=1.0,
                               bias_scale=1.0,
                               yspread=1.0,
                      M=100,y0range=[-1,1],y1range=[-1,1],
                     size=400.0, linewidth=5.0,
                    steps=100, batchsize=10, eta=0.1,
                              random_init=False,
                              visualize_nsteps=1,
                              plot_target=True):
    """
    Visualize the training of a neural network.

    weights, biases, and activations define the neural network
    (the starting point of the optimization; for the detailed description,
    see the help for visualize_network)

    If you want to have layers randomly initialized, just provide
    the number of neurons for each layer as 'num_neurons'. This should include
    all layers, including input (2 neurons) and output (1), so num_neurons=[2,3,5,4,1] is
    a valid example. In this case, weight_scale and bias_scale define the
    spread of the random Gaussian variables used to initialize all weights and biases.

    target_function is the name of the function that we
    want to approximate; it must be possible to
    evaluate this function on a batch of samples, by
    calling target_function(y) on an array y of
    shape [batchsize,2], where
    the second index refers to the two coordinates
    (input neuron values) y0 and y1. The return
    value must be an array with one index, corresponding
    to the batchsize. A valid example is:

    def my_target(y):
        return( np.sin(y[:,0]) + np.cos(y[:,1]) )

    steps is the number of training steps

    batchsize is the number of samples per training step

    eta is the learning rate (stepsize in the gradient descent)

    yspread denotes the spread of the Gaussian
    used to sample points in (y0,y1)-space

    visualize_n_steps>1 means skip some steps before
    visualizing again (can speed up things)

    plot_target=True means do plot the target function in a corner

    For all the other parameters, see the help for
        visualize_network

    weights and biases as given here will be used
    as starting points, unless you specify
    random_init=True, in which case they will be
    used to determine the spread of Gaussian random
    variables used for initialization!
    """

    if num_neurons is not None: # build weight matrices as randomly initialized
        weights=[weight_scale*np.random.randn(num_neurons[j+1],num_neurons[j]) for j in range(len(num_neurons)-1)]
        biases=[bias_scale*np.random.randn(num_neurons[j+1]) for j in range(len(num_neurons)-1)]

    swapped_weights=[]
    for j in range(len(weights)):
        swapped_weights.append(np.transpose(weights[j]))
    init_layer_variables(swapped_weights,biases,activations)

    if plot_target:
        y0,y1=np.meshgrid(np.linspace(y0range[0],y0range[1],M),np.linspace(y1range[0],y1range[1],M))
        y=np.zeros([M*M,2])
        y[:,0]=y0.flatten()
        y[:,1]=y1.flatten()
        plot_target_values=np.reshape(target_function(y),[M,M])
    else:
        plot_target_values=None

    y_target=np.zeros([batchsize,1])
    costs=np.zeros(steps)

    for j in range(steps):
        # produce samples (random points in y0,y1-space):
        y_in=yspread*np.random.randn(batchsize,2)
        # apply target function to those points:
        y_target[:,0]=target_function(y_in)
        # do one training step on this batch of samples:
        costs[j]=train_net(y_in,y_target,eta)

        # now visualize the updated network:
        if j%visualize_nsteps==0:
            #clear_output(wait=True) # for animation, but bad in colab.
            if j>10:
                cost_max=np.average(costs[0:j])*1.5
            else:
                cost_max=costs[0]
            visualize_network(Weights,Biases,activations,
                          M,y0range=y0range,y1range=y1range,
                         size=size, linewidth=linewidth,
                             weights_are_swapped=True,
                             layers_already_initialized=True,
                             plot_cost_function=costs,
                             current_cost=costs[j],
                             cost_max=cost_max,
                             plot_target=plot_target_values)
            sleep(0.1) # wait a bit before next step (probably not needed)

### Example 1: Training the AND gate.

We think of $\text{AND}:\mathbb{R}^2 \to \mathbb{R}$ as $\theta(y^0 + y^1)$, which outputs $1$ if the argument is positive and $0$ else. This is AND because we are thinking of $\mathbb{Z}_2$ as $\{-1,1\}$, with $-1$ akin to logical zero and $1$ akin to logical one. Kind of stupid, but whatever. As a colourmap, this should look like a sharp line $y^1=-y^0$ with output $+1$ above it, and output $0$ below it. We see how this is trained below.

In [ ]:
# @title
def my_target(y):
    return( 1.0*( (y[:,0]+y[:,1])>0) ) #the inputs are thought of as +- 1, hence in this case y_0 + y_1 =1 means and == true. the 1.0* part just makes it a float instead of a pure integer. The target is really the theta function Theta(y0 + y1):R^2 -> R^1. This will look like the line y1=-y0 in the plane, with the upper right half being +1 and the lower left half being 0.

visualize_network_training(weights=[ [
    [0.2,-0.9]  # initialisation weights of 2 input neurons for single output neuron
    ] ],
    biases=[
        [0.0] # bias for single output neuron
            ],
    target_function=my_target, # the target function to approximate
    activations=[ 'sigmoid' # activation for output
                ],
    y0range=[-3,3],y1range=[-3,3],
    steps=1000, eta=5.0, batchsize=200,
                          visualize_nsteps=10, plot_target=True) #we have a batch of 200 inputs where y0 goes from -3 to 3, and y1 too. in each step of the training, we compute the outputs according to the current theta, use backprop to compute the derivatives for each C_x(theta), then use stochastic GD with the same batch of inputs to correct the thetas, then go again! the graph on the bottom left shows (average) cost as a function of step size i.e. a function of theta in the path in the parametetr space.

Now we have trained a single layer NN to compute AND, let's start developing some multi-layer networks. Below is the quarter-space AND function. This is the function $\theta(y^0)\cdot \theta(y^1)$ that is one when both $y^1>0$ and $y^0>0$, zero else. Logically speaking, this is the same function on $\mathbb{Z}_2$. However, it looks different when viewed as a function on $\mathbb{R}^2$. See below.

In [ ]:
# @title
def my_target(y):
    return( 1.0* (y[:,0]> 0) * (y[:,1]>0) )

visualize_network_training(weights=[ [
    [0.2,-0.9],[0.3,0.4]  # weights of 2 input neurons for single output neuron
    ],
    [  [0.2,0.5]  ]
    ],
    biases=[
        [0.1,-0.2],
        [0.0] # bias for single output neuron
            ],
    target_function=my_target, # the target function to approximate
    activations=[ 'sigmoid',
                 'sigmoid' #two layers, so two sigmoids!
                ],
    y0range=[-3,3],y1range=[-3,3],
    yspread=3,
    steps=1000, eta=5.0, batchsize=200,
                          visualize_nsteps=10)

We now again train towards the function $\theta(y^0)\theta(y^1)$ but this time with randomly initialised weights in the neurons so that we don't have to put it in by hand - this is built in to the functions developed above. We also see how RelU fares against the sigmoid I have been using so far. The idea is that RelU is a gamble: if I initialise close to the true answer then it is very efficient (no exponentials for python to compute), but if I'm far from the solution then the gradient descent just doesn't work a lot of the time. I'll think of this like the "strong force" of machine learning, akin to QCD behaviour.

In [ ]:
# @title
def my_target(y):
    return( 1.0* (y[:,0]> 0) * (y[:,1]>0) )

visualize_network_training(weights=[],biases=[],
    num_neurons=[2,10,5,1], # this generates randomly initialized layers of the given neuron numbers!
    bias_scale=0.0, weight_scale=0.1, # the scale of the random numbers
    target_function=my_target, # the target function to approximate
    activations=[ 'reLU',
                 'reLU',
                 'reLU'  #3 layers, so 3 activations
                ],
    y0range=[-3,3],y1range=[-3,3],
    yspread=3,
    steps=2000, eta=.1, batchsize=200,
                          visualize_nsteps=100)

Next, we train XOR, which makes contact with HW1. We view it as +1 for $y^0 y^1 \lt 0$ and 0 else. Yes, that's a fucking dumb way to view it, but again it makes sense if $-1 \sim 0$ and $+1 \sim 1$ where the right side is the bit. It will look like $+1$ in the upper left and bottom right quadrants of the plane, and $0$ in the other two quadrants.

So we define it as

    def weirdXOR(y):
      return 1.0*( y[:,0]*y[:,1] < 0 )

which acts on a whole batch at once, computing a truth value and then converting it to a float.

How do we lay out the network? We can experiment with different layer numbers. I think sigmoid activation will do because this function isn't so crazy that I need an efficient convergence.


HINT: If you cannot get it to work properly, these parameter values worked for me, for a single hidden layer of 5 neurons and sigmoids:

steps=20000, eta=1.0, batchsize=200, visualize_nsteps=100

In [ ]:
# @title
def weirdXOR(y):
      return 1.0*( y[:,0]*y[:,1] < 0 )


visualize_network_training(weights=[],biases=[],
    num_neurons=[2,10,1], # this generates randomly initialized layers of the given neuron numbers!
    bias_scale=0.0, weight_scale=0.1, # the scale of the random numbers
    target_function=weirdXOR, # the target function to approximate
    activations=[ 'sigmoid',
                 'sigmoid'  #2 layers, so 2 activations
                ],
    y0range=[-3,3],y1range=[-3,3],
    yspread=3,
    steps=20000, eta=1.0, batchsize=200,
                          visualize_nsteps=1000)

How does the learning success depend on these choices and the learning parameters (like $\eta$)? It looks like large step sizes get the job done here, because the gradient of the cost function is really not very high at all here. Because we don't have a very steep potential, barely anything changes for small $\eta$.

Let's train the NN to converge to some more interesting functions. I want to see some circular stuff, so let's try
$$
r^2={(y^0)^2 + (y^1)^2}.
$$
which is accomplished by

    def rsquare(y):
      return y[:,0]**2 + y[:,1]**2

This is interesting because the value just increases as we go out to infinity, so the cost function involves a lot of different points on the plane. Let's see what happens.

In [ ]:
# @title
def rsquare(y):
  return y[:,0]**2 + y[:,1]**2


visualize_network_training(weights=[],biases=[],
    num_neurons=[2,5,10,5,1], # this generates randomly initialized layers of the given neuron numbers!
    bias_scale=0.5, weight_scale=0.5, # the scale of the random numbers
    target_function=rsquare, # the target function to approximate
    activations=[ 'reLU',
                  'sigmoid',
                  'sigmoid',
                  'reLU'
                ],
    y0range=[-3,3],y1range=[-3,3],
    yspread=3,
    steps=20000, eta=1, batchsize=500,
                          visualize_nsteps=1000)

It's pretty hard to fit! reLU seems to just bring me along this path in parameter space where we have the constant function, at least around the origin! That's interesting... the cost of a constant function $f$ against the target $r^2$ is $f^2 -2fr^2 + r^4$. There is a minima to this potential, and somehow reLU is not taking us off the subspace of constant functions! Or more likely, since the cost is still changing in each step, we are evaluating differences at larger and larger radii which dominate the cost since $C \sim r^4$, so this is a really bad example I chose because the dynamics is dominated by the point at infinity. Good, I learned something.

Let's try a step function that is equal to $1$ inside the unit sphere and 0 else.

In [ ]:
# @title
def unitsphere(y):
  return 1.0* (y[:,0]**2 + y[:,1]**2 < 1 )

visualize_network_training(weights=[],biases=[],
    num_neurons=[2,10,10,1], # this generates randomly initialized layers of the given neuron numbers!
    bias_scale=0.5, weight_scale=0.5, # the scale of the random numbers
    target_function=unitsphere, # the target function to approximate
    activations=[ 'sigmoid',
                 'sigmoid',
                  'sigmoid'
                ],
    y0range=[-3,3],y1range=[-3,3],
    yspread=3,
    steps=20000, eta=1, batchsize=200,
                          visualize_nsteps=1000)

Report: Fuck yes! Looks great! This works much better than the $r^2$ function I tried above because it has compact support, so all the computations essentially focus on how the function looks in this interesting vicinity near the origin!

Let's try a heart shape:

In [ ]:
# @title
def heart(y):
    a=0.8; r=0.5; R=2.0
    return( 1.0*( (y[:,0]**2 + y[:,1]**2 -1)**3 -y[:,0]**2 * y[:,1]**3 <0 )  )

visualize_network_training(weights=[],biases=[],
    num_neurons=[2,20,20,1], # this generates randomly initialized layers of the given neuron numbers!
    bias_scale=0.5, weight_scale=0.5, # the scale of the random numbers
    target_function=heart, # the target function to approximate
    activations=[ 'sigmoid',
                 'reLU',
                  'sigmoid'
                ],
    y0range=[-3,3],y1range=[-3,3],
    yspread=3,
    steps=20000, eta=0.5, batchsize=500,
                          visualize_nsteps=1000)

### (3) Analyze the evolution of the slope $w$ during stochastic gradient descent on a cost function given by $C=(1/2) \sum_j (w x_j - \tilde{w} x_j)^2$, where $x_j$ are the $N$ samples drawn from a Gaussian distribution in a single training step. (this is an advanced exercise)

This is not a very well-defined question, so let's turn it into something that makes sense. Suppose the parameter space is $\Theta=\mathbb{R}^2$ and so whatever our neural network is depends on the two parameters $\theta^a$ (it could still have many weights and biases, but they are collectively a function of $\theta$). An input $\vec{x}$ is associated to a cost $C_x(\theta) := \frac{1}{2}\sum_{j=1}^{N}(\theta^0 - \theta^1)^2 (x^j)^2$.

We now endow the input space $\mathbb{R}^N$ with a probability measure $\mathbb{P}$ which is Gaussian, which is to say the density function is
\begin{align}
p(\vec{x})=\frac{1}{((2\pi)^N \det \Sigma)^{\frac{1}{2}}} \exp \bigg( -\frac{1}{2}(\vec{x}-\vec{\mu}) \Sigma^{-1} (\vec{x}-\vec{\mu}) \bigg)
\end{align}
for some invertible $\Sigma$ whose eigenvalues are essentially $\sigma^2$, the covariance. So this structure tells us how likely certain inputs are. We then see that $C_{-}(\theta)$ is a random variable on $\mathbb{R}^N$ and that $C_x(-) \in C^\infty(\Theta)$. The global cost function $C \in C^\infty(\Theta)$ is then
\begin{align}
C(\theta)=\mathbb{E}[C_{-}(\theta)] = \int_{\mathbb{R}^N} \text{d}^N x ~p(x) C_x(\theta)
\end{align}
which is what we want to perform gradient descent against. To perform stochastic descent, let's pick a batch i.e. an event $B \subseteq \mathbb{R}^N$. We then form a cost with respect to this event as
\begin{align}
C_B(\theta) = \mathbb{E}[C_{-}(\theta)|B] &= \int_{\mathbb{R}^N} \text{d}^N x ~p(x|B) C_x(\theta)
\\
&=\frac{1}{\text{vol}(B)} \int_{B}\text{d}^N x~ p(x)C_x(\theta)
\end{align}
and have the descent formula
\begin{align}
\delta \theta^a = - \eta \partial^a C(\theta) = -\eta \partial^a C_B(\theta) + \text{noise}
\end{align}
where repeatedly sampling from these batches will cause the errors to cancel out on average.

So we want to do this and plot how $\theta$ evolves! Plan:

- Initialise phase space $\Theta$ and a point $\theta$.
- Randomly pick a bunch of inputs using the Gaussian above to make up a finite batch $B=\{\vec{x}_1,...,\vec{x}_N \}$.
- Compute $\frac{1}{|B|} \sum_{\vec{x} \in B} C_\vec{x}(\theta)$, which approximates the theoretical $C_B(\theta)$.
- Update $\theta$ to first order in the derivative of this term, and store the new value
- Pick a new batch $B'$ and start again, now at $\theta+\delta \theta$. Update the values of $theta$ and store them.
- Once we have a list of points on the plane, plot them together with lines connecting adjacent points.
- Repeat this for different sizes of $B$ being sampled, and plot everything at once. We expect curves generated by larger batches to be smoother, and curves generated by small batches to be more jaggedy, but globally correct.
